<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 9: </b>LangServe와 평가</h2>
<br>

## LangServe 서버 설정

이 노트북은 LangChain과 [**LangServe**](https://github.com/langchain-ai/langserve)를 사용해 대화형 웹 애플리케이션을 개발하는 데 관심 있는 분들을 위한 놀이터입니다. 웹 애플리케이션 맥락에서 LangChain의 가능성을 보여 주는 최소한의 코드 예제를 제공하는 것이 목적입니다.

이 섹션은 LangChain의 Runnable 인터페이스와 FastAPI를 사용해 간단한 API 서버를 구성하는 과정을 안내합니다. 예제는 `ChatNVIDIA` 같은 LangChain 모델을 통합하여 접근 가능한 API 라우트를 만들고 배포하는 방법을 보여 줍니다. 이를 통해 프론트엔드 서비스의 [**`frontend_server.py`**](./frontend/frontend_server.py) 세션에 기능을 제공할 수 있으며, 이 세션은 다음을 강하게 기대합니다:
- 기본 챗봇을 위한 `:9012/basic_chat`이라는 간단한 엔드포인트(아래 예시).
- RAG 챗봇을 위한 `:9012/retriever`와 `:9012/generator` 엔드포인트 쌍.
- 최종 평가에 필요한 **Evaluate** 유틸리티를 위한 위 세 가지 모두. *자세한 내용은 뒤에서!*

**중요 참고 사항:**
- 활성 FastAPI 셀을 종료하려면 사각형( $\square$ ) 버튼을 두 번 클릭하세요. 첫 번째 클릭은 무시되거나 비동기 프로세스의 try-catch 루틴을 트리거할 수 있습니다.
- 그래도 동작하지 않으면 **Kernel -> Restart Kernel**로 이 노트북을 하드 재시작하세요.
- 셀에서 FastAPI 서버가 실행 중이면 그 프로세스가 이 노트북을 블로킹합니다. 다른 노트북은 영향을 받지 않아야 합니다. 

<br>

### **Part 1:** /basic_chat 엔드포인트 제공하기

`/basic_chat` 엔드포인트를 독립 Python 파일로 실행하기 위한 안내가 제공됩니다. 이는 프론트엔드에서 내부 추론 없이 기본적인 결정을 내리는 데 사용됩니다.

In [ ]:
%%writefile server_app.py
# https://python.langchain.com/docs/langserve#server
from fastapi import FastAPI
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langserve import add_routes

## May be useful later
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.prompt_values import ChatPromptValue
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnablePassthrough
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_community.document_transformers import LongContextReorder
from functools import partial
from operator import itemgetter

from langchain_community.vectorstores import FAISS

## TODO: Make sure to pick your LLM and do your prompt engineering as necessary for the final assessment
embedder = NVIDIAEmbeddings(
    model="course/embedding",
    base_url="http://llm_client:9000/v1",
)

# In Colab, replace the service client above with:
# from langchain_community.embeddings import FastEmbedEmbeddings
# embedder = FastEmbedEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

app = FastAPI(
  title="LangChain Server",
  version="1.0",
  description="A simple api server using Langchain's Runnable interfaces",
)

## PRE-ASSESSMENT: Run as-is and see the basic chain in action

add_routes(
    app,
    instruct_llm,
    path="/basic_chat",
)

## ASSESSMENT TODO: Implement these components as appropriate

add_routes(
    app,
    RunnableLambda(lambda x: "Not Implemented"),
    path="/generator",
)

add_routes(
    app,
    RunnableLambda(lambda x: []),
    path="/retriever",
)

## Might be encountered if this were for a standalone python file...
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=9012)

In [ ]:
## Works, but will block the notebook.
!python server_app.py  

## Will technically work, but not recommended in a notebook. 
## You may be surprised at the interesting side effects...
# import os
# os.system("python server_app.py &")

<br>

### **Part 2:** 서버 사용하기:

Google Colab에서는 쉽게 활용할 수 없지만(적어도 많은 특수한 트릭 없이는), 위 스크립트는 노트북 프로세스에 묶인 실행 중인 서버를 유지합니다. 서버가 실행 중인 동안에는 (서비스 종료/재시작을 제외하고) 이 노트북을 사용하려 하지 마세요.

하지만 다른 파일에서는 다음 인터페이스를 사용해 `basic_chat` 엔드포인트에 접근할 수 있습니다:

```python
from langserve import RemoteRunnable
from langchain_core.output_parsers import StrOutputParser

llm = RemoteRunnable("http://0.0.0.0:9012/basic_chat/") | StrOutputParser()
for token in llm.stream("Hello World! How is it going?"):
    print(token, end='')
```

**다른 파일에서 직접 시도해 보고 동작하는지 확인하세요!**

<br>

### **Part 3: 최종 평가**

**이 노트북은 최종 평가를 완료하는 데 사용됩니다!** 코스를 모두 마쳤다면, 이 노트북을 복제하고 프론트엔드를 새 탭에서 연 뒤, 위의 `/generator`와 `/retriever` 엔드포인트를 구현하여 Evaluate 기능을 완성하는 것을 권장합니다! 프론트엔드로 가는 빠른 링크는 아래 셀을 실행하세요:

<a href="/8090" target="_blank" style="display: inline-block; padding: 12px 24px; background-color: #76b900; color: white; text-decoration: none; border-radius: 4px; font-weight: bold; margin: 3px;">Gradio Frontend UI (<code>/8090</code> -> <code>:8090</code> 안정성을 위해)</a>

In [ ]:
# %%js
# // Manual Access Without NGINX
# var url = 'http://'+window.location.host+':8090';
# element.innerHTML = '<a style="color:green;" target="_blank" href='+url+'><h1>< Link To Gradio Frontend ></h1></a>';

<hr>
<br>

#### **평가 힌트:** 
다음 기능은 프론트엔드 마이크로서비스에 이미 구현되어 있습니다. 

```python
## Necessary Endpoints
chains_dict = {
    'basic' : RemoteRunnable("http://lab:9012/basic_chat/"),
    'retriever' : RemoteRunnable("http://lab:9012/retriever/"),  ## For the final assessment
    'generator' : RemoteRunnable("http://lab:9012/generator/"),  ## For the final assessment
}

basic_chain = chains_dict['basic']

## Retrieval-Augmented Generation Chain

retrieval_chain = (
    {'input' : (lambda x: x)}
    | RunnableAssign(
        {'context' : itemgetter('input') 
        | chains_dict['retriever'] 
        | LongContextReorder().transform_documents
        | docs2str
    })
)

output_chain = RunnableAssign({"output" : chains_dict['generator'] }) | output_puller
rag_chain = retrieval_chain | output_chain
```

**이 엔드포인트 수집 전략에 맞추려면, 파이프라인 기능을 중복 구현하지 말고 빠져 있는 기능만 배포하세요!**

----

<div style="width: 55%%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>